In [1]:
from datasets import load_dataset
from dotenv import load_dotenv
import os
import psycopg

load_dotenv()

True

In [2]:
import sys
import os


# Get the absolute path to the parent directory
parent_dir = os.path.abspath(os.path.join(os.getcwd(), '..'))

# Add the parent directory to sys.path if it's not already there
if parent_dir not in sys.path:
    sys.path.append(parent_dir)

## Define Variables

In [ ]:

DATASET_SOURCE = 'suniltvl/ragbench'
DATA_SPLIT = 'test'
VECTOR_DATABASES = ['chroma'] # ['chroma', 'milvus']
EMBEDDING_MODELS = ["BAAI/LLM-Embedder", "BAAI/bge-large-en-v1.5"]
MAX_CHUNKS = 5000
CHUNKING_SIZES = [256, 512, 1024]
CHUNKING_OVERLAPS = [50, 100, 200]
SEPARATORS = ["\n\n", "\n", " ", ".", ","]
DOMAINS = {
    'cs':
    {
        "delucionqa":"Jeep manual", 
        "emanual": "TV manual", 
        "techqa":"Technotes"
    },
    'gk':
    {
        "hotpotqa":"wiki 1",
        "msmarco":"web pages",
        "hagrid":"wiki 2",
        "expertqa":"googlesearch"
    }
}
DATABASE_URL = os.getenv("LOCAL_POSTGRE_DATABASE_URL")



In [4]:
from utils.helper import get_questions_table_name

print(get_questions_table_name(DOMAINS))

qtn_cs_gk


## Prepare Questions Database

In [5]:

def get_questions_table_name():
    """
    Get the questions table name.
    
    Returns:
        str: The questions table name
    """
    questions_table = "qtn"

    domains_name = ""
    for domain in DOMAINS.keys():
        domains_name += f"_{domain}"

    questions_table += domains_name
    
    return questions_table


def get_load_docs(db_name: str):
    dataset = load_dataset(DATASET_SOURCE, db_name, split=DATA_SPLIT)
    
    return dataset

def deduplicate_data(data, doc_type):
    data_dict = {}
    for d in data:
        document = " ".join(d["documents"])
        if document in data_dict:
            data_dict[document]["docid"].append(d["id"])
        else:
            data_dict[document] = {"docid":[d["id"]]}
            
    for k in data_dict:
        data_dict[k]["document_type"] = doc_type
        
    return data_dict

### Add questions to database

In [6]:
def add_questions_to_db(dataset, DOMAIN, DATASET_NAME, TABLE_NAME):
    try:
        # conn = sqlite3.connect(sqldb)
        conn = psycopg.connect(DATABASE_URL)
        cursor = conn.cursor()
        quest_dict = dict()
        # j = 1
        
        for d in dataset:
            # print(d)
            # print(quest_dict)
            # for ds in dataset:
            #     if ds["question"] == d["question"]:
            if d["question"] in quest_dict:
                if d["generation_model_name"].startswith("gpt"):
                    quest_dict[d["question"]].update({
                        "gpt_adherence": d["adherence_score"],
                        "gpt_relevance_score": d["relevance_score"],
                        "gpt_utilization_score": d["utilization_score"],
                        "gpt_completeness_score": d["completeness_score"]
                    })
                elif d["generation_model_name"].startswith("claude"):
                    quest_dict[d["question"]].update({
                        "claude_adherence": d["adherence_score"],
                        "claude_relevance_score": d["relevance_score"],
                        "claude_utilization_score": d["utilization_score"],
                        "claude_completeness_score": d["completeness_score"]
                    })
            else:
                if d["generation_model_name"].startswith("gpt"):
                    quest_dict[d["question"]] = {
                        "gpt_adherence": d["adherence_score"],
                        "gpt_relevance_score": d["relevance_score"],
                        "gpt_utilization_score": d["utilization_score"],
                        "gpt_completeness_score": d["completeness_score"]
                    }
                elif d["generation_model_name"].startswith("claude"):
                    quest_dict[d["question"]] = {
                        "claude_adherence": d["adherence_score"],
                        "claude_relevance_score": d["relevance_score"],
                        "claude_utilization_score": d["utilization_score"],
                        "claude_completeness_score": d["completeness_score"]
            }
                    
            # if j == sample:
            #     break
            # else:
            #     j += 1

        # return quest_dict
        questions_count = len(quest_dict.keys())
        i = 1
        # base_session_id = uuid.uuid4()
        for q, v in quest_dict.items():
            # print(q)

            # response, sent_list = self.simple_rag(query=q, expert_domain=domain,
            #                                      retriever=retriever, gen_model=gen_model,
            #                                      temperature=temperature)


            domain_name = DOMAIN 
            dataset_name = DATASET_NAME
            query = q
            gpt_adherence = v.get("gpt_adherence")
            gpt_relevance = v.get("gpt_relevance_score")
            gpt_utilization = v.get("gpt_utilization_score")
            gpt_completeness = v.get("gpt_completeness_score")
            claude_adherence = v.get("claude_adherence")
            claude_relevance = v.get("claude_relevance_score")
            claude_utilization = v.get("claude_utilization_score")
            claude_completeness = v.get("claude_completeness_score") 

            sql_query = f"""
                INSERT INTO {TABLE_NAME} (
                    domain_name, dataset_name, query, gpt_adherence, gpt_relevance, gpt_utilization,
                    gpt_completeness, claude_adherence, claude_relevance, claude_utilization, claude_completeness
                ) VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
            """

            values = (
            domain_name,
            dataset_name,
            query,
            gpt_adherence,
            gpt_relevance,
            gpt_utilization,
            gpt_completeness,
            claude_adherence,
            claude_relevance,
            claude_utilization,
            claude_completeness
            )
            cursor.execute(sql_query, values)
            conn.commit()
            print(f"{i}/{questions_count} completed")
            i += 1
            # time.sleep(10)

        conn.close()

        print(f"All entries inserted")

    except Exception as e:
        print(e)
        if 'conn' in locals():
            conn.close()

### Loop Domains

In [7]:
def loop_domains():   

    table_name = get_questions_table_name()
    # Loop through domains and datasets
    for domain_short_name, data_set in DOMAINS.items():

        domain_name = domain_short_name.upper()
        data_set_name = ""

        print(f"Processing domain: {domain_short_name}")        
        # Loop through datasets
        for data_set_path, name in data_set.items():

            data_set_name = name

            print(f"  Processing dataset: {data_set_path} > {name}")
            
            # Read from dataset (HuggingFace)
            docs = get_load_docs(data_set_path)

            print(f"  Total documents: {len(docs)}")

            add_questions_to_db(docs, domain_name, data_set_name, table_name)

            

            

In [18]:
loop_domains()

Processing domain: cs
  Processing dataset: delucionqa > Jeep manual
  Total documents: 184
1/92 completed
2/92 completed
3/92 completed
4/92 completed
5/92 completed
6/92 completed
7/92 completed
8/92 completed
9/92 completed
10/92 completed
11/92 completed
12/92 completed
13/92 completed
14/92 completed
15/92 completed
16/92 completed
17/92 completed
18/92 completed
19/92 completed
20/92 completed
21/92 completed
22/92 completed
23/92 completed
24/92 completed
25/92 completed
26/92 completed
27/92 completed
28/92 completed
29/92 completed
30/92 completed
31/92 completed
32/92 completed
33/92 completed
34/92 completed
35/92 completed
36/92 completed
37/92 completed
38/92 completed
39/92 completed
40/92 completed
41/92 completed
42/92 completed
43/92 completed
44/92 completed
45/92 completed
46/92 completed
47/92 completed
48/92 completed
49/92 completed
50/92 completed
51/92 completed
52/92 completed
53/92 completed
54/92 completed
55/92 completed
56/92 completed
57/92 completed
58/92

In [ ]:
# dataset = load_dataset(DATASET_SOURCE, DATASET_NAME, split=DATA_SPLIT)

In [ ]:
# try:
#     # conn = sqlite3.connect(sqldb)
#     conn = psycopg.connect(DATABASE_URL)
#     cursor = conn.cursor()
#     quest_dict = dict()
#     # j = 1
    
#     for d in dataset:
#         # print(d)
#         # print(quest_dict)
#         # for ds in dataset:
#         #     if ds["question"] == d["question"]:
#         if d["question"] in quest_dict:
#             if d["generation_model_name"].startswith("gpt"):
#                 quest_dict[d["question"]].update({
#                     "gpt_adherence": d["adherence_score"],
#                     "gpt_relevance_score": d["relevance_score"],
#                     "gpt_utilization_score": d["utilization_score"],
#                     "gpt_completeness_score": d["completeness_score"]
#                 })
#             elif d["generation_model_name"].startswith("claude"):
#                 quest_dict[d["question"]].update({
#                     "claude_adherence": d["adherence_score"],
#                     "claude_relevance_score": d["relevance_score"],
#                     "claude_utilization_score": d["utilization_score"],
#                     "claude_completeness_score": d["completeness_score"]
#                 })
#         else:
#             if d["generation_model_name"].startswith("gpt"):
#                 quest_dict[d["question"]] = {
#                     "gpt_adherence": d["adherence_score"],
#                     "gpt_relevance_score": d["relevance_score"],
#                     "gpt_utilization_score": d["utilization_score"],
#                     "gpt_completeness_score": d["completeness_score"]
#                 }
#             elif d["generation_model_name"].startswith("claude"):
#                 quest_dict[d["question"]] = {
#                     "claude_adherence": d["adherence_score"],
#                     "claude_relevance_score": d["relevance_score"],
#                     "claude_utilization_score": d["utilization_score"],
#                     "claude_completeness_score": d["completeness_score"]
#         }
                
#         # if j == sample:
#         #     break
#         # else:
#         #     j += 1

#     # return quest_dict
#     questions_count = len(quest_dict.keys())
#     i = 1
#     # base_session_id = uuid.uuid4()
#     for q, v in quest_dict.items():
#         # print(q)

#         # response, sent_list = self.simple_rag(query=q, expert_domain=domain,
#         #                                      retriever=retriever, gen_model=gen_model,
#         #                                      temperature=temperature)


#         domain_name = DOMAIN 
#         dataset_name = DATASET_NAME
#         query = q
#         gpt_adherence = v.get("gpt_adherence")
#         gpt_relevance = v.get("gpt_relevance_score")
#         gpt_utilization = v.get("gpt_utilization_score")
#         gpt_completeness = v.get("gpt_completeness_score")
#         claude_adherence = v.get("claude_adherence")
#         claude_relevance = v.get("claude_relevance_score")
#         claude_utilization = v.get("claude_utilization_score")
#         claude_completeness = v.get("claude_completeness_score") 

#         sql_query = f"""
#             INSERT INTO {TABLE_NAME} (
#                 domain_name, dataset_name, query, gpt_adherence, gpt_relevance, gpt_utilization,
#                 gpt_completeness, claude_adherence, claude_relevance, claude_utilization, claude_completeness
#             ) VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
#         """

#         values = (
#         domain_name,
#         dataset_name,
#         query,
#         gpt_adherence,
#         gpt_relevance,
#         gpt_utilization,
#         gpt_completeness,
#         claude_adherence,
#         claude_relevance,
#         claude_utilization,
#         claude_completeness
#         )
#         cursor.execute(sql_query, values)
#         conn.commit()
#         print(f"{i}/{questions_count} completed")
#         i += 1
#         # time.sleep(10)

#     conn.close()

#     print(f"All entries inserted")

# except Exception as e:
#     print(e)
#     if 'conn' in locals():
#         conn.close()

In [ ]:
print("hi")